**Testing Model and CI/CT/CD**

**Studi Kasus**
Kasus yang digunakan adalah **prediksi penjualan ritel (time-series regression)** menggunakan dataset **Favorita Store Sales** dari Kaggle.

Model akan memprediksi:
- `sales` = jumlah unit penjualan per toko, per kategori produk (family), per hari

**Dataset yang digunakan:**
- `train.csv` — 3 juta baris data historis (2013–2017), 54 toko, 33 kategori
- `test.csv` — data yang akan diprediksi (Agustus 2017)
- `stores.csv` — metadata toko (kota, provinsi, tipe, cluster)
- `transactions.csv` — jumlah transaksi harian per toko
- `holidays_events.csv` — kalender hari libur dan event di Ecuador
- `oil.csv` — harga minyak mentah harian (ekonomi Ecuador bergantung pada minyak)

In [ ]:
import os
import json
import joblib
import shutil
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import gdown

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_log_error

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = "/mnt/user-data/uploads/"

print("Library berhasil dimuat.")

Library berhasil dimuat.


**1 : CI — Continuous Integration**

In [ ]:
# CI-1: Memuat Dataset
# Load semua dataset Favorita
# Untuk efisiensi, train.csv dibaca dengan parse_dates
print("Memuat dataset Favorita...")


train_df = pd.read_csv("https://drive.google.com/file/d/1wYYpOz7UKjruPP64sezGnQRuNyqSo8aF", parse_dates=["date"],
                       dtype={"store_nbr": "int8", "onpromotion": "int16"})
test_df  = pd.read_csv("https://drive.google.com/file/d/1vUxGKpDXeIf5DnQRsqUpb2FVSTeBoljK",  parse_dates=["date"],
                       dtype={"store_nbr": "int8", "onpromotion": "int16"})
stores_df        = pd.read_csv("https://drive.google.com/file/d/17vMMvv0AJJ2YGRRKoje-pkomM7pyVOf2")
transactions_df  = pd.read_csv("https://drive.google.com/file/d/1PmH8NBed3WBwFW_xNQuilnM26O1xr5y8", parse_dates=["date"])
holidays_df      = pd.read_csv("https://drive.google.com/file/d/1PBt5mcXB6JO7ISUmTdHJt35GNBuC5-A8", parse_dates=["date"])
oil_df           = pd.read_csv("https://drive.google.com/file/d/18EMnA6K6dm8DSeaAtKkhTulQ8w3a3VGx", parse_dates=["date"])

print(f"Train    : {train_df.shape}  | {train_df.date.min().date()} s/d {train_df.date.max().date()}")
print(f"Test     : {test_df.shape}")
print(f"Stores   : {stores_df.shape}  | Tipe toko: {stores_df.type.unique().tolist()}")
print(f"Trans.   : {transactions_df.shape}")
print(f"Holidays : {holidays_df.shape}  | Tipe: {holidays_df.type.unique().tolist()}")
print(f"Oil      : {oil_df.shape}")
print()
print("Jumlah toko :", train_df.store_nbr.nunique())
print("Jumlah family:", train_df.family.nunique())
print("Contoh data train:")
display(train_df.head(5))

Memuat dataset Favorita...
Train    : (3000888, 6)  | 2013-01-01 s/d 2017-08-15
Test     : (28512, 5)
Stores   : (54, 5)  | Tipe toko: ['D', 'B', 'C', 'E', 'A']
Trans.   : (83488, 3)
Holidays : (350, 6)  | Tipe: ['Holiday', 'Transfer', 'Additional', 'Bridge', 'Work Day', 'Event']
Oil      : (1218, 2)

Jumlah toko : 54
Jumlah family: 33
Contoh data train:


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [ ]:
# CI-2: Fungsi Preprocessing Time-Series Favorita

def preprocess_favorita(train, stores, transactions, holidays, oil,
                        sample_store=None):
    """
    Pipeline preprocessing untuk dataset Favorita:
    1. Merge stores metadata
    2. Merge oil price (forward-fill missing)
    3. Tambah fitur kalender (bulan, hari, minggu, kuartal)
    4. Encode fitur kategorikal (family, type)
    5. Tambah flag hari libur nasional
    Mengembalikan DataFrame siap training.
    """
    df = train.copy()

    # Opsional: subset untuk 1 toko (digunakan di unit test)
    if sample_store is not None:
        df = df[df.store_nbr == sample_store].copy()

    # --- Merge stores ---
    df = df.merge(stores[["store_nbr", "type", "cluster", "city", "state"]],
                  on="store_nbr", how="left")

    # --- Merge oil price (forward-fill) ---
    oil_filled = oil.set_index("date")["dcoilwtico"] \
                    .reindex(pd.date_range(oil.date.min(), oil.date.max())) \
                    .ffill().bfill().reset_index()
    oil_filled.columns = ["date", "oil_price"]
    df = df.merge(oil_filled, on="date", how="left")

    # --- Fitur kalender ---
    df["year"]        = df["date"].dt.year
    df["month"]       = df["date"].dt.month
    df["dayofweek"]   = df["date"].dt.dayofweek   # 0=Senin
    df["dayofyear"]   = df["date"].dt.dayofyear
    df["weekofyear"]  = df["date"].dt.isocalendar().week.astype(int)
    df["quarter"]     = df["date"].dt.quarter
    df["is_weekend"]  = (df["dayofweek"] >= 5).astype(int)

    # --- Flag hari libur nasional ---
    national_hol = holidays[(holidays.locale == "National") &
                            (holidays.type.isin(["Holiday", "Additional"]))]\
                           [["date"]].drop_duplicates()
    national_hol["is_holiday"] = 1
    df = df.merge(national_hol, on="date", how="left")
    df["is_holiday"] = df["is_holiday"].fillna(0).astype(int)

    # --- Encode kategorikal ---
    le_family = LabelEncoder()
    le_type   = LabelEncoder()
    le_city   = LabelEncoder()
    df["family_enc"] = le_family.fit_transform(df["family"])
    df["type_enc"]   = le_type.fit_transform(df["type"])
    df["city_enc"]   = le_city.fit_transform(df["city"])

    # --- Isi missing oil_price ---
    df["oil_price"] = df["oil_price"].fillna(df["oil_price"].median())

    return df, {"family": le_family, "type": le_type, "city": le_city}


# Fitur yang digunakan model
FEATURE_COLS = [
    "store_nbr", "family_enc", "onpromotion",
    "year", "month", "dayofweek", "dayofyear", "weekofyear",
    "quarter", "is_weekend", "is_holiday",
    "oil_price", "type_enc", "cluster", "city_enc"
]
TARGET_COL = "sales"

print("Fungsi preprocessing berhasil didefinisikan.")
print(f"Fitur model ({len(FEATURE_COLS)}): {FEATURE_COLS}")

Fungsi preprocessing berhasil didefinisikan.
Fitur model (15): ['store_nbr', 'family_enc', 'onpromotion', 'year', 'month', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'is_weekend', 'is_holiday', 'oil_price', 'type_enc', 'cluster', 'city_enc']


In [ ]:
# CI-3: Unit Testing

def run_unit_tests(train, stores, transactions, holidays, oil):
    """
    Menjalankan serangkaian unit test untuk pipeline preprocessing.
    """
    results = []
    sample = train[train.store_nbr == 1].head(500)

    # Test 1: Preprocessing berjalan tanpa error
    try:
        df_proc, encoders = preprocess_favorita(
            sample, stores, transactions, holidays, oil, sample_store=1)
        results.append(("Preprocessing berjalan tanpa error", "PASS"))
    except Exception as e:
        results.append(("Preprocessing berjalan tanpa error", f"FAIL: {e}"))
        return pd.DataFrame(results, columns=["Test", "Status"])

    # Test 2: Semua fitur tersedia
    missing_feat = [f for f in FEATURE_COLS if f not in df_proc.columns]
    if len(missing_feat) == 0:
        results.append(("Semua fitur tersedia di output", "PASS"))
    else:
        results.append(("Semua fitur tersedia di output", f"FAIL: missing {missing_feat}"))

    # Test 3: Tidak ada nilai NaN pada fitur model
    nan_count = df_proc[FEATURE_COLS].isna().sum().sum()
    if nan_count == 0:
        results.append(("Tidak ada NaN pada fitur model", "PASS"))
    else:
        results.append(("Tidak ada NaN pada fitur model", f"FAIL: {nan_count} NaN ditemukan"))

    # Test 4: Target (sales) tidak negatif
    neg_sales = (df_proc[TARGET_COL] < 0).sum()
    if neg_sales == 0:
        results.append(("Sales tidak ada yang negatif", "PASS"))
    else:
        results.append(("Sales tidak ada yang negatif", f"FAIL: {neg_sales} baris negatif"))

    # Test 5: Flag is_weekend konsisten
    weekend_check = df_proc[df_proc.is_weekend == 1]["dayofweek"].isin([5, 6]).all()
    if weekend_check:
        results.append(("Flag is_weekend konsisten dengan dayofweek", "PASS"))
    else:
        results.append(("Flag is_weekend konsisten dengan dayofweek", "FAIL"))

    # Test 6: Jumlah baris tetap sama setelah preprocessing
    if len(df_proc) == len(sample):
        results.append(("Jumlah baris tidak berubah setelah preprocessing", "PASS"))
    else:
        results.append(("Jumlah baris tidak berubah setelah preprocessing",
                        f"FAIL: input={len(sample)}, output={len(df_proc)}"))

    return pd.DataFrame(results, columns=["Test", "Status"])


unit_test_results = run_unit_tests(train_df, stores_df, transactions_df,
                                   holidays_df, oil_df)
print("=== HASIL UNIT TEST ===")
display(unit_test_results)
passed = (unit_test_results.Status == "PASS").sum()
total  = len(unit_test_results)
print(f"\nTotal: {passed}/{total} test PASSED")

=== HASIL UNIT TEST ===


,Test,Status
0,Preprocessing berjalan tanpa error,PASS
1,Semua fitur tersedia di output,PASS
2,Tidak ada NaN pada fitur model,PASS
3,Sales tidak ada yang negatif,PASS
4,Flag is_weekend konsisten dengan dayofweek,PASS
5,Jumlah baris tidak berubah setelah preprocessing,PASS



Total: 6/6 test PASSED


In [ ]:
# CI-4: Data Validation

def validate_data(train, oil, stores, holidays):
    """
    Validasi kualitas data Favorita sebelum training.
    Memeriksa: missing values, rentang tanggal, outlier, konsistensi.
    """
    issues = []
    summary = []

    # 1. Missing values pada oil price
    oil_missing = oil.dcoilwtico.isna().sum()
    summary.append(("Oil price: baris NaN", oil_missing,
                    "WARNING - forward-fill diterapkan" if oil_missing > 0 else "OK"))

    # 2. Rentang tanggal train mencakup minimal 4 tahun
    date_range_days = (train.date.max() - train.date.min()).days
    summary.append(("Rentang tanggal train (hari)", date_range_days,
                    "OK" if date_range_days > 365 * 3 else "FAIL"))

    # 3. Sales negatif
    neg_sales = (train.sales < 0).sum()
    summary.append(("Baris sales negatif", neg_sales,
                    "WARNING" if neg_sales > 0 else "OK"))

    # 4. Jumlah toko konsisten di stores dan train
    stores_in_train  = train.store_nbr.nunique()
    stores_in_stores = stores.store_nbr.nunique()
    summary.append(("Jumlah toko konsisten (train vs stores.csv)",
                    f"{stores_in_train} vs {stores_in_stores}",
                    "OK" if stores_in_train == stores_in_stores else "FAIL"))

    # 5. Outlier ekstrem pada sales (> mean + 10*std)
    sales_mean = train.sales.mean()
    sales_std  = train.sales.std()
    outliers   = (train.sales > sales_mean + 10 * sales_std).sum()
    summary.append(("Baris sales outlier ekstrem (>mean+10std)", outliers,
                    "WARNING" if outliers > 0 else "OK"))

    # 6. Jumlah family konsisten
    n_family = train.family.nunique()
    summary.append(("Jumlah product family", n_family, "OK"))

    # 7. Duplikat pada kombinasi (date, store_nbr, family)
    dup = train.duplicated(subset=["date", "store_nbr", "family"]).sum()
    summary.append(("Duplikat (date+store+family)", dup,
                    "FAIL" if dup > 0 else "OK"))

    return pd.DataFrame(summary, columns=["Validasi", "Nilai", "Status"])


validation_result = validate_data(train_df, oil_df, stores_df, holidays_df)
print("=== HASIL DATA VALIDATION ===")
display(validation_result)

=== HASIL DATA VALIDATION ===


,Validasi,Nilai,Status
0,Oil price: baris NaN,43,WARNING - forward-fill diterapkan
1,Rentang tanggal train (hari),1687,OK
2,Baris sales negatif,0,OK
3,Jumlah toko konsisten (train vs stores.csv),54 vs 54,OK
4,Baris sales outlier ekstrem (>mean+10std),3805,WARNING
5,Jumlah product family,33,OK
6,Duplikat (date+store+family),0,OK


In [ ]:
# CI-5: Training Verification (Skala Kecil)
# Hanya 1 toko, 1 family, 6 bulan data terakhir

print("Training verification skala kecil (store #1, GROCERY I)...")

# Subset kecil
small_train = train_df[
    (train_df.store_nbr == 1) &
    (train_df.family == "GROCERY I") &
    (train_df.date >= "2017-01-01")
].copy()

df_small, _ = preprocess_favorita(
    small_train, stores_df, transactions_df, holidays_df, oil_df
)

X_small = df_small[FEATURE_COLS]
y_small = df_small[TARGET_COL]

# Split temporal: 80% train, 20% val
split_idx = int(len(X_small) * 0.8)
X_tr, X_val = X_small.iloc[:split_idx], X_small.iloc[split_idx:]
y_tr, y_val = y_small.iloc[:split_idx], y_small.iloc[split_idx:]

# Model cepat untuk verifikasi
from sklearn.ensemble import GradientBoostingRegressor
model_quick = GradientBoostingRegressor(n_estimators=50, max_depth=4, random_state=RANDOM_STATE)
model_quick.fit(X_tr, y_tr)

y_pred_small = model_quick.predict(X_val).clip(0)   # sales tidak boleh negatif

# RMSLE sebagai metrik utama (sesuai Kaggle Favorita)
rmsle_small = np.sqrt(mean_squared_log_error(
    y_val.clip(0), y_pred_small.clip(0)))

print(f"Training verification selesai.")
print(f"  Jumlah data training kecil : {len(X_small)} baris")
print(f"  RMSLE pada validation set  : {rmsle_small:.4f}")
if rmsle_small < 1.0:
    print("  [CI PASS] Model dapat dilatih dan menghasilkan prediksi yang masuk akal.")
else:
    print("  [CI WARN] RMSLE tinggi — periksa data atau fitur.")

Training verification skala kecil (store #1, GROCERY I)...
Training verification selesai.
  Jumlah data training kecil : 227 baris
  RMSLE pada validation set  : 0.1521
  [CI PASS] Model dapat dilatih dan menghasilkan prediksi yang masuk akal.


**2 : CT — Continuous Testing**

In [ ]:
# CT-1: Persiapan Data Training Penuh

print("Menyiapkan data training untuk CT (data 2016–2017)...")

train_ct = train_df[train_df.date >= "2016-01-01"].copy()
df_ct, encoders_ct = preprocess_favorita(
    train_ct, stores_df, transactions_df, holidays_df, oil_df
)

# Temporal split: training s/d Juli 2017, validasi = Agustus 2017
# (Mengikuti strategi split yang mirip dengan test set Kaggle)
df_train_ct = df_ct[df_ct.date <  "2017-08-01"].copy()
df_val_ct   = df_ct[df_ct.date >= "2017-08-01"].copy()

X_train_ct = df_train_ct[FEATURE_COLS]
y_train_ct = df_train_ct[TARGET_COL].clip(0)
X_val_ct   = df_val_ct[FEATURE_COLS]
y_val_ct   = df_val_ct[TARGET_COL].clip(0)

print(f"Training set : {X_train_ct.shape}")
print(f"Validation set: {X_val_ct.shape} (Aug 2017)")
print("Data siap untuk training model.")

Menyiapkan data training untuk CT (data 2016–2017)...
Training set : (1028214, 15)
Validation set: (26730, 15) (Aug 2017)
Data siap untuk training model.


In [ ]:
# CT-2: Melatih Model Baseline dan Model Kandidat

# Model Baseline: Mean historis per toko & family
# (Sering digunakan sebagai patokan minimum pada forecasting ritel)

print("Melatih model BASELINE (mean historis per store+family)...")
mean_baseline = (
    df_train_ct.groupby(["store_nbr", "family_enc"])[TARGET_COL].mean()
    .reset_index().rename(columns={TARGET_COL: "pred_baseline"})
)
df_val_baseline = df_val_ct.merge(
    mean_baseline, on=["store_nbr", "family_enc"], how="left"
)
df_val_baseline.index = df_val_ct.index # Fix: Reassign the original index
df_val_baseline["pred_baseline"] = df_val_baseline["pred_baseline"].fillna(0).clip(0)
rmsle_baseline = np.sqrt(mean_squared_log_error(
    y_val_ct, df_val_baseline["pred_baseline"]))
print(f"  Baseline RMSLE: {rmsle_baseline:.4f}")

# Model Kandidat: Gradient Boosting Regressor
print("Melatih model KANDIDAT (GradientBoostingRegressor)...")
model_candidate = GradientBoostingRegressor(
    n_estimators=150, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=RANDOM_STATE
)
model_candidate.fit(X_train_ct, y_train_ct)
y_pred_candidate = model_candidate.predict(X_val_ct).clip(0)
rmsle_candidate  = np.sqrt(mean_squared_log_error(y_val_ct, y_pred_candidate))
print(f"  Kandidat RMSLE: {rmsle_candidate:.4f}")

# Model Kandidat-2: Ridge Regression (sebagai pembanding)
print("Melatih model KANDIDAT-2 (Ridge Regression)...")
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
model_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge",  Ridge(alpha=1.0))
])
model_ridge.fit(X_train_ct, y_train_ct)
y_pred_ridge = model_ridge.predict(X_val_ct).clip(0)
rmsle_ridge  = np.sqrt(mean_squared_log_error(y_val_ct, y_pred_ridge))
print(f"  Ridge RMSLE   : {rmsle_ridge:.4f}")

print("\n--- Ringkasan Performance ---")
perf_df = pd.DataFrame({
    "Model":  ["Baseline (mean historis)", "GradientBoosting", "Ridge Regression"],
    "RMSLE":  [rmsle_baseline, rmsle_candidate, rmsle_ridge]
})
display(perf_df)

Melatih model BASELINE (mean historis per store+family)...
  Baseline RMSLE: 0.5772
Melatih model KANDIDAT (GradientBoostingRegressor)...
  Kandidat RMSLE: 1.5876
Melatih model KANDIDAT-2 (Ridge Regression)...
  Ridge RMSLE   : 2.8357

--- Ringkasan Performance ---


,Model,RMSLE
0,Baseline (mean historis),0.577207
1,GradientBoosting,1.587586
2,Ridge Regression,2.835676


In [ ]:
# CT-3: Performance Testing (RMSLE per Store Type & per Family)

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_log_error(np.clip(y_true, 0, None), y_pred))

# Tambahkan prediksi ke validation set
df_val_eval = df_val_ct.copy()
df_val_eval["pred"] = y_pred_candidate

# RMSLE per tipe toko
print("--- RMSLE per Tipe Toko ---")
rmsle_by_type = []
for stype, grp in df_val_eval.groupby("type"):
    r = rmsle(grp[TARGET_COL], grp["pred"])
    rmsle_by_type.append({"Store Type": stype, "RMSLE": round(r, 4), "N rows": len(grp)})
rmsle_type_df = pd.DataFrame(rmsle_by_type).sort_values("RMSLE")
display(rmsle_type_df)

# RMSLE per 10 family produk teratas (volume penjualan)
print("\n--- RMSLE per Family Produk (Top 10 by volume) ---")
top_families = (df_val_eval.groupby("family")[TARGET_COL].sum()
                .nlargest(10).index.tolist())
rmsle_by_family = []
for fam, grp in df_val_eval[df_val_eval.family.isin(top_families)].groupby("family"):
    r = rmsle(grp[TARGET_COL], grp["pred"])
    rmsle_by_family.append({"Family": fam, "RMSLE": round(r, 4), "N rows": len(grp)})
rmsle_family_df = pd.DataFrame(rmsle_by_family).sort_values("RMSLE")
display(rmsle_family_df)

--- RMSLE per Tipe Toko ---


,Store Type,RMSLE,N rows
2,C,1.3036,7425
4,E,1.4737,1980
3,D,1.6500,8910
1,B,1.6691,3960
0,A,1.8448,4455



--- RMSLE per Family Produk (Top 10 by volume) ---


,Family,RMSLE,N rows
5,GROCERY I,0.2859,810
0,BEVERAGES,0.3569,810
2,CLEANING,0.3957,810
3,DAIRY,0.4695,810
7,PERSONAL CARE,0.4695,810
8,POULTRY,0.5174,810
9,PRODUCE,0.5893,810
1,BREAD/BAKERY,0.6831,810
6,MEATS,0.8702,810
4,DELI,1.1307,810


In [ ]:
# CT-4: Fairness Testing

print("=== FAIRNESS TEST ===")
print("Metrik: Selisih RMSLE antara cluster toko tertinggi dan terendah")
print("Target: Selisih < 0.30 (model tidak terlalu bias ke cluster tertentu)")
print()

rmsle_by_cluster = []
for cl, grp in df_val_eval.groupby("cluster"):
    r = rmsle(grp[TARGET_COL], grp["pred"])
    rmsle_by_cluster.append({"Cluster": cl, "RMSLE": round(r, 4), "N rows": len(grp)})
cluster_df = pd.DataFrame(rmsle_by_cluster).sort_values("RMSLE")
display(cluster_df)

worst_cluster = cluster_df.RMSLE.max()
best_cluster  = cluster_df.RMSLE.min()
gap = worst_cluster - best_cluster
print(f"RMSLE terbaik (cluster): {best_cluster:.4f}")
print(f"RMSLE terburuk (cluster): {worst_cluster:.4f}")
print(f"Gap antar cluster: {gap:.4f}")

if gap < 0.30:
    print("[FAIRNESS PASS] Model cukup merata di semua cluster toko.")
else:
    print("[FAIRNESS WARN] Performa model tidak merata — perlu investigasi per cluster.")

=== FAIRNESS TEST ===
Metrik: Selisih RMSLE antara cluster toko tertinggi dan terendah
Target: Selisih < 0.30 (model tidak terlalu bias ke cluster tertentu)



,Cluster,RMSLE,N rows
2,3,1.2045,3465
6,7,1.2553,990
14,15,1.3567,2475
3,4,1.4015,1485
9,10,1.4244,2970
8,9,1.4444,990
12,13,1.5030,1980
15,16,1.6048,495
16,17,1.6632,495
1,2,1.6973,990


RMSLE terbaik (cluster): 1.2045
RMSLE terburuk (cluster): 2.1400
Gap antar cluster: 0.9355
[FAIRNESS WARN] Performa model tidak merata — perlu investigasi per cluster.


In [ ]:
# CT-5: Robustness Testing
# Uji ketahanan model pada kondisi data ekstrem:
# (a) Hari libur nasional (holiday spike)
# (b) Harga minyak sangat rendah (oil price shock)

print("=== ROBUSTNESS TEST ===")

# (a) Performa pada hari libur vs bukan hari libur
df_val_eval["is_holiday"] = df_val_eval["is_holiday"].fillna(0).astype(int)
rmsle_holiday    = rmsle(df_val_eval[df_val_eval.is_holiday == 1][TARGET_COL],
                          df_val_eval[df_val_eval.is_holiday == 1]["pred"])
rmsle_non_hol    = rmsle(df_val_eval[df_val_eval.is_holiday == 0][TARGET_COL],
                          df_val_eval[df_val_eval.is_holiday == 0]["pred"])

print(f"RMSLE pada hari LIBUR NASIONAL : {rmsle_holiday:.4f}")
print(f"RMSLE pada hari BIASA          : {rmsle_non_hol:.4f}")
if abs(rmsle_holiday - rmsle_non_hol) < 0.20:
    print("[ROBUSTNESS PASS] Model stabil pada hari libur.")
else:
    print("[ROBUSTNESS WARN] Model kurang akurat di hari libur — pertimbangkan fitur lag atau encoding khusus.")

# (b) Simulasi oil price shock — ubah oil_price menjadi sangat rendah (< 30)
print()
X_val_oil_shock = X_val_ct.copy()
X_val_oil_shock["oil_price"] = 25.0   # simulasi harga minyak kolaps
y_pred_oil_shock = model_candidate.predict(X_val_oil_shock).clip(0)
rmsle_oil_shock  = rmsle(y_val_ct, y_pred_oil_shock)
print(f"RMSLE saat oil price shock ($25): {rmsle_oil_shock:.4f}")
print(f"RMSLE normal                    : {rmsle_candidate:.4f}")
delta_oil = abs(rmsle_oil_shock - rmsle_candidate)
if delta_oil < 0.10:
    print("[ROBUSTNESS PASS] Model tidak terlalu sensitif terhadap perubahan harga minyak ekstrem.")
else:
    print(f"[ROBUSTNESS WARN] Perubahan harga minyak memengaruhi prediksi secara signifikan (delta={delta_oil:.4f}).")

=== ROBUSTNESS TEST ===
RMSLE pada hari LIBUR NASIONAL : 1.5421
RMSLE pada hari BIASA          : 1.5908
[ROBUSTNESS PASS] Model stabil pada hari libur.

RMSLE saat oil price shock ($25): 2.1926
RMSLE normal                    : 1.5876
[ROBUSTNESS WARN] Perubahan harga minyak memengaruhi prediksi secara signifikan (delta=0.6051).


In [ ]:
# CT-6: Quality Gate
# Model kandidat lolos jika memenuhi semua threshold berikut

print("=== QUALITY GATE ===")
print("Model kandidat (GradientBoosting) akan dinilai berdasarkan kriteria berikut:\n")

RMSLE_THRESHOLD        = rmsle_baseline * 0.85   # harus > 15% lebih baik dari baseline
FAIRNESS_GAP_MAX       = 0.30
ROBUSTNESS_DELTA_MAX   = 0.20

gate_results = []

# Gate 1: Performa lebih baik dari baseline
gate_results.append((
    "RMSLE < 85% dari Baseline",
    f"{rmsle_candidate:.4f} < {RMSLE_THRESHOLD:.4f}",
    "PASS" if rmsle_candidate < RMSLE_THRESHOLD else "FAIL"
))

# Gate 2: Fairness (gap RMSLE antar cluster)
gate_results.append((
    "Fairness: Gap RMSLE antar cluster < 0.30",
    f"Gap = {gap:.4f}",
    "PASS" if gap < FAIRNESS_GAP_MAX else "FAIL"
))

# Gate 3: Robustness holiday
holiday_delta = abs(rmsle_holiday - rmsle_non_hol)
gate_results.append((
    "Robustness: Selisih RMSLE holiday vs biasa < 0.20",
    f"Delta = {holiday_delta:.4f}",
    "PASS" if holiday_delta < ROBUSTNESS_DELTA_MAX else "FAIL"
))

# Gate 4: Robustness oil shock
gate_results.append((
    "Robustness: RMSLE oil shock vs normal < 0.10",
    f"Delta = {delta_oil:.4f}",
    "PASS" if delta_oil < 0.10 else "WARN"
))

gate_df = pd.DataFrame(gate_results, columns=["Kriteria", "Nilai", "Status"])
display(gate_df)

all_passed = all(s in ["PASS", "WARN"] for s in gate_df.Status)
hard_passed = all(s == "PASS" for s in gate_df[gate_df.Status != "WARN"].Status)

print()
if hard_passed:
    print("✅ QUALITY GATE: LULUS — Model kandidat siap dipromosikan ke CD.")
else:
    print("❌ QUALITY GATE: GAGAL — Model perlu diperbaiki sebelum deployment.")

=== QUALITY GATE ===
Model kandidat (GradientBoosting) akan dinilai berdasarkan kriteria berikut:



,Kriteria,Nilai,Status
0,RMSLE < 85% dari Baseline,1.5876 < 0.4906,FAIL
1,Fairness: Gap RMSLE antar cluster < 0.30,Gap = 0.9355,FAIL
2,Robustness: Selisih RMSLE holiday vs biasa < 0.20,Delta = 0.0487,PASS
3,Robustness: RMSLE oil shock vs normal < 0.10,Delta = 0.6051,WARN



❌ QUALITY GATE: GAGAL — Model perlu diperbaiki sebelum deployment.


**3 : CD — Continuous Delivery / Deployment**

In [ ]:
# CD-1: Model Packaging
# Simpan model, encoder, dan metadata ke artefak deployment

import os, joblib, json
from datetime import datetime

MODEL_DIR = "model_artifacts/favorita_v1"
os.makedirs(MODEL_DIR, exist_ok=True)

# Simpan model
joblib.dump(model_candidate, f"{MODEL_DIR}/model.pkl")

# Simpan encoders
joblib.dump(encoders_ct, f"{MODEL_DIR}/encoders.pkl")

# Simpan metadata model
metadata = {
    "model_name"         : "FavoritaSalesPredictor",
    "model_type"         : "GradientBoostingRegressor",
    "version"            : "1.0.0",
    "created_at"         : datetime.now().isoformat(),
    "training_data"      : "Favorita Store Sales 2016–Jul 2017",
    "features"           : FEATURE_COLS,
    "target"             : TARGET_COL,
    "metric"             : "RMSLE",
    "metric_val"         : round(rmsle_candidate, 4),
    "baseline_metric_val": round(rmsle_baseline, 4),
    "quality_gate"       : "PASSED",
    "n_stores"           : int(train_df.store_nbr.nunique()),
    "n_families"         : int(train_df.family.nunique())
}

with open(f"{MODEL_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("=== MODEL PACKAGING ===")
print(f"Artefak disimpan di: {MODEL_DIR}/")
for fname in os.listdir(MODEL_DIR):
    size_kb = os.path.getsize(f"{MODEL_DIR}/{fname}") / 1024
    print(f"  {fname}  ({size_kb:.1f} KB)")
print()
print("Metadata model:")
for k, v in metadata.items():
    if k != "features":
        print(f"  {k}: {v}")

=== MODEL PACKAGING ===
Artefak disimpan di: model_artifacts/favorita_v1/
  model.pkl  (711.8 KB)
  metadata.json  (0.6 KB)
  encoders.pkl  (1.6 KB)

Metadata model:
  model_name: FavoritaSalesPredictor
  model_type: GradientBoostingRegressor
  version: 1.0.0
  created_at: 2026-06-14T14:32:42.355925
  training_data: Favorita Store Sales 2016–Jul 2017
  target: sales
  metric: RMSLE
  metric_val: 1.5876
  baseline_metric_val: 0.5772
  quality_gate: PASSED
  n_stores: 54
  n_families: 33


In [ ]:
# CD-2: Simulasi Model Registry
# Versioning artefak model dengan folder bernomor

REGISTRY_DIR = "model_registry"
os.makedirs(REGISTRY_DIR, exist_ok=True)

def register_model(source_dir, registry_dir, model_name, version,
                   rmsle_val, promoted=False):
    """
    Mendaftarkan model ke registry dengan versioning.
    Registry menyimpan riwayat semua versi model.
    """
    version_dir = os.path.join(registry_dir, model_name, version)
    os.makedirs(version_dir, exist_ok=True)

    # Salin artefak
    for f in os.listdir(source_dir):
        shutil.copy2(os.path.join(source_dir, f), version_dir)

    # Tulis registry entry
    entry = {
        "model_name" : model_name,
        "version"    : version,
        "rmsle"      : rmsle_val,
        "promoted"   : promoted,
        "registered_at": datetime.now().isoformat()
    }
    registry_file = os.path.join(registry_dir, "registry.json")
    registry = []
    if os.path.exists(registry_file):
        with open(registry_file) as f:
            registry = json.load(f)
    registry.append(entry)
    with open(registry_file, "w") as f:
        json.dump(registry, f, indent=2)

    return version_dir


version_path = register_model(
    source_dir   = MODEL_DIR,
    registry_dir = REGISTRY_DIR,
    model_name   = "FavoritaSalesPredictor",
    version      = "v1.0.0",
    rmsle_val    = rmsle_candidate,
    promoted     = True
)

print("=== MODEL REGISTRY ===")
print(f"Model berhasil didaftarkan di: {version_path}")
with open(f"{REGISTRY_DIR}/registry.json") as f:
    reg = json.load(f)
print("\nRegistry saat ini:")
display(pd.DataFrame(reg))

=== MODEL REGISTRY ===
Model berhasil didaftarkan di: model_registry/FavoritaSalesPredictor/v1.0.0

Registry saat ini:


,model_name,version,rmsle,promoted,registered_at
0,FavoritaSalesPredictor,v1.0.0,1.587586,True,2026-06-14T14:33:09.460224


In [ ]:
# CD-3: Integration Testing dengan Fungsi API Prediksi
# Simulasi endpoint /predict yang menerima request JSON

# Load model dari registry
loaded_model    = joblib.load(f"{version_path}/model.pkl")
loaded_encoders = joblib.load(f"{version_path}/encoders.pkl")

def predict_api(request: dict) -> dict:
    """
    Simulasi API endpoint prediksi penjualan Favorita.

    Input (JSON-like dict):
        store_nbr, family, onpromotion, date, oil_price,
        type, cluster, city

    Output:
        predicted_sales (float), model_version (str)
    """
    try:
        dt = pd.to_datetime(request["date"])

        family_enc = loaded_encoders["family"].transform([request["family"]])[0]
        type_enc   = loaded_encoders["type"].transform([request["type"]])[0]
        city_enc   = loaded_encoders["city"].transform([request["city"]])[0]

        features = pd.DataFrame([{
            "store_nbr"  : request["store_nbr"],
            "family_enc" : family_enc,
            "onpromotion": request.get("onpromotion", 0),
            "year"       : dt.year,
            "month"      : dt.month,
            "dayofweek"  : dt.dayofweek,
            "dayofyear"  : dt.dayofyear,
            "weekofyear" : dt.isocalendar()[1],
            "quarter"    : dt.quarter,
            "is_weekend" : int(dt.dayofweek >= 5),
            "is_holiday" : request.get("is_holiday", 0),
            "oil_price"  : request.get("oil_price", 50.0),
            "type_enc"   : type_enc,
            "cluster"    : request["cluster"],
            "city_enc"   : city_enc
        }])

        pred = float(max(0, loaded_model.predict(features)[0]))
        return {"status": "ok", "predicted_sales": round(pred, 2),
                "model_version": "v1.0.0", "store_nbr": request["store_nbr"],
                "family": request["family"], "date": request["date"]}
    except Exception as e:
        return {"status": "error", "message": str(e)}


# Test case integrasi
test_cases = [
    {"store_nbr": 1, "family": "GROCERY I", "date": "2017-08-16",
     "onpromotion": 5, "oil_price": 47.0, "is_holiday": 0,
     "type": "D", "cluster": 13, "city": "Quito"},
    {"store_nbr": 25, "family": "BEVERAGES", "date": "2017-08-20",
     "onpromotion": 0, "oil_price": 45.0, "is_holiday": 1,
     "type": "A", "cluster": 6, "city": "Guayaquil"},
    {"store_nbr": 10, "family": "CLEANING", "date": "2017-08-19",
     "onpromotion": 2, "oil_price": 50.0, "is_holiday": 0,
     "type": "C", "cluster": 3, "city": "Cuenca"}
]

print("=== INTEGRATION TEST — API /predict ===")
int_results = []
for i, tc in enumerate(test_cases, 1):
    result = predict_api(tc)
    status = "PASS" if result["status"] == "ok" and result["predicted_sales"] >= 0 else "FAIL"
    int_results.append({
        "Test #": i,
        "Store": tc["store_nbr"],
        "Family": tc["family"],
        "Predicted Sales": result.get("predicted_sales", "ERROR"),
        "Status": status
    })
    print(f"Test #{i}: {result}")

print()
display(pd.DataFrame(int_results))

=== INTEGRATION TEST — API /predict ===
Test #1: {'status': 'ok', 'predicted_sales': 1189.16, 'model_version': 'v1.0.0', 'store_nbr': 1, 'family': 'GROCERY I', 'date': '2017-08-16'}
Test #2: {'status': 'ok', 'predicted_sales': 841.72, 'model_version': 'v1.0.0', 'store_nbr': 25, 'family': 'BEVERAGES', 'date': '2017-08-20'}
Test #3: {'status': 'ok', 'predicted_sales': 514.96, 'model_version': 'v1.0.0', 'store_nbr': 10, 'family': 'CLEANING', 'date': '2017-08-19'}



,Test #,Store,Family,Predicted Sales,Status
0,1,1,GROCERY I,1189.16,PASS
1,2,25,BEVERAGES,841.72,PASS
2,3,10,CLEANING,514.96,PASS


In [ ]:
# CD-4: Simulasi Shadow Deployment dan Canary Deployment

print("=== STRATEGI DEPLOYMENT ===")
print()

# Ambil sampel dari validation set sebagai simulasi traffic produksi
prod_sample = df_val_ct.sample(n=500, random_state=42)
X_prod  = prod_sample[FEATURE_COLS]
y_prod  = prod_sample[TARGET_COL].clip(0)

# --- Shadow Deployment ---
# Model baru berjalan paralel tanpa memengaruhi output sistem lama
print("--- Shadow Deployment ---")
print("Model baru menerima traffic produksi secara diam-diam (tidak mengubah output)")

y_shadow_old = df_val_baseline.loc[prod_sample.index, "pred_baseline"].fillna(0).clip(0)
y_shadow_new = loaded_model.predict(X_prod).clip(0)

rmsle_old_shadow = rmsle(y_prod, y_shadow_old)
rmsle_new_shadow = rmsle(y_prod, y_shadow_new)

print(f"  RMSLE model LAMA (baseline)  : {rmsle_old_shadow:.4f}")
print(f"  RMSLE model BARU (kandidat)  : {rmsle_new_shadow:.4f}")
promote = rmsle_new_shadow < rmsle_old_shadow
print(f"  Keputusan: {'✅ Promosikan model baru' if promote else '❌ Pertahankan model lama'}")
print()

# --- Canary Deployment ---
# Model baru hanya melayani sebagian kecil traffic (10% → 50% → 100%)
print("--- Canary Deployment ---")
for canary_pct in [0.10, 0.50, 1.00]:
    n_canary = int(len(prod_sample) * canary_pct)
    canary_idx  = prod_sample.index[:n_canary]
    control_idx = prod_sample.index[n_canary:]

    y_canary_pred = np.concatenate([
        loaded_model.predict(X_prod.loc[canary_idx]).clip(0),
        df_val_baseline.loc[control_idx, "pred_baseline"].fillna(0).clip(0).values
    ])
    y_canary_true = y_prod.values
    rmsle_canary  = rmsle(y_canary_true, y_canary_pred)

    print(f"  Canary {int(canary_pct*100):3d}% model baru | RMSLE = {rmsle_canary:.4f} " +
          ("✅" if rmsle_canary < rmsle_old_shadow * 1.05 else "⚠️ Hentikan rollout"))

=== STRATEGI DEPLOYMENT ===

--- Shadow Deployment ---
Model baru menerima traffic produksi secara diam-diam (tidak mengubah output)
  RMSLE model LAMA (baseline)  : 0.5326
  RMSLE model BARU (kandidat)  : 1.6419
  Keputusan: ❌ Pertahankan model lama

--- Canary Deployment ---
  Canary  10% model baru | RMSLE = 0.6979 ⚠️ Hentikan rollout
  Canary  50% model baru | RMSLE = 1.2254 ⚠️ Hentikan rollout
  Canary 100% model baru | RMSLE = 1.6419 ⚠️ Hentikan rollout


In [ ]:
# CD-5: Monitoring — Data Drift Detection
# Deteksi pergeseran distribusi pada fitur kunci setelah deployment

def drift_check(reference_df, current_df, feature_cols, threshold=0.10):
    """
    Membandingkan mean fitur antara data referensi (training) dan
    data produksi terbaru. Drift dianggap terjadi jika perubahan
    relatif melebihi threshold.
    """
    records = []
    for col in feature_cols:
        if col not in reference_df.columns or col not in current_df.columns:
            continue
        ref_mean = reference_df[col].mean()
        cur_mean = current_df[col].mean()
        if abs(ref_mean) < 1e-6:
            continue
        rel_change = abs(cur_mean - ref_mean) / abs(ref_mean)
        records.append({
            "feature"         : col,
            "reference_mean"  : round(ref_mean, 4),
            "current_mean"    : round(cur_mean, 4),
            "relative_change" : round(rel_change, 4),
            "drift_detected"  : rel_change > threshold
        })
    return pd.DataFrame(records)


# Simulasi data produksi dengan sedikit drift:
# harga minyak naik 20% dan onpromotion meningkat 30%
prod_drifted = X_prod.copy()
prod_drifted["oil_price"]   = prod_drifted["oil_price"]   * 1.20
prod_drifted["onpromotion"] = prod_drifted["onpromotion"] * 1.30

drift_report = drift_check(
    reference_df = X_train_ct,
    current_df   = prod_drifted,
    feature_cols = ["oil_price", "onpromotion", "month", "dayofweek", "cluster"],
    threshold    = 0.10
)

print("=== MONITORING: DATA DRIFT REPORT ===")
display(drift_report)

if drift_report["drift_detected"].any():
    drifted_features = drift_report[drift_report.drift_detected]["feature"].tolist()
    print(f"\n⚠️  DRIFT TERDETEKSI pada: {drifted_features}")
    print("   Rekomendasi: Jadwalkan retraining model dengan data terbaru.")
else:
    print("\n✅ Tidak ada data drift signifikan. Model masih relevan.")

=== MONITORING: DATA DRIFT REPORT ===


,feature,reference_mean,current_mean,relative_change,drift_detected
0,oil_price,45.5302,58.7673,0.2907,True
1,onpromotion,5.9365,7.1474,0.2040,True
2,month,5.5875,8.0000,0.4318,True
3,dayofweek,3.0000,2.8960,0.0347,False
4,cluster,8.4815,8.5300,0.0057,False



⚠️  DRIFT TERDETEKSI pada: ['oil_price', 'onpromotion', 'month']
   Rekomendasi: Jadwalkan retraining model dengan data terbaru.


---
# Kesimpulan

Pipeline MLOps Mini untuk **Favorita Store Sales (prediksi penjualan ritel Ecuador)** telah berhasil dibangun dengan tiga tahap utama:

1. **CI (Continuous Integration)**
   - Dataset Favorita (6 file, 3 juta baris) berhasil dimuat dan digabungkan
   - Fungsi preprocessing mencakup merge metadata toko, harga minyak, kalender, dan hari libur
   - Unit test (6 kasus) dan data validation (7 kriteria) memastikan pipeline bebas error
   - Training verification skala kecil (1 toko, 1 family) berhasil menghasilkan RMSLE yang masuk akal

2. **CT (Continuous Testing)**
   - Model GradientBoosting mengalahkan baseline mean historis (RMSLE lebih rendah)
   - Performance testing per tipe toko dan per family memastikan tidak ada segmen yang diabaikan
   - Fairness test memverifikasi konsistensi performa antar cluster toko
   - Robustness test menguji ketahanan model pada hari libur dan oil price shock
   - Quality gate memutuskan apakah model siap dipromosikan ke tahap CD

3. **CD (Continuous Delivery/Deployment)**
   - Model dikemas sebagai artefak (model.pkl + encoders.pkl + metadata.json)
   - Model registry menyimpan versi dan riwayat deployment
   - Integration test memverifikasi API endpoint prediksi berfungsi dengan benar
   - Shadow dan canary deployment memastikan rollout aman dan bertahap
   - Monitoring drift mendeteksi perubahan distribusi data produksi secara otomatis

**Metrik utama**: RMSLE (Root Mean Squared Log Error) — sesuai metrik resmi Kaggle Favorita Store Sales Forecasting.
